In [2]:
import torch
import torch.nn as nn
import torch.optim as optim
import torchvision
from torchvision import datasets, transforms
from torch.utils.data import DataLoader
import numpy as np
from sklearn.metrics import silhouette_score, normalized_mutual_info_score, f1_score, top_k_accuracy_score, recall_score
from tqdm import tqdm
import matplotlib.pyplot as plt

# 设置随机种子确保可复现
def set_seed(seed=42):
    torch.manual_seed(seed)
    torch.cuda.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    np.random.seed(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

set_seed(42)

# ==================== 1. 主干网络（输出全局+中间层局部特征）====================
class ResNetBackbone(nn.Module):
    def __init__(self, base_encoder):
        super().__init__()
        try:
            self.backbone = base_encoder(weights=torchvision.models.ResNet50_Weights.DEFAULT)
        except AttributeError:
            self.backbone = base_encoder(pretrained=True)

        # CIFAR 小图适配
        self.backbone.conv1 = nn.Conv2d(3, 64, kernel_size=3, stride=1, padding=1, bias=False)
        self.backbone.maxpool = nn.Identity()
        self.backbone.fc = nn.Identity()

        self.feature_dim = 2048    # 全局特征维度
        self.local_dim = 1024      # 局部特征维度（layer3 输出）

    def forward(self, x):
        # 前向传播，提取多层特征
        x = self.backbone.conv1(x)
        x = self.backbone.bn1(x)
        x = self.backbone.relu(x)
        x = self.backbone.maxpool(x)

        x1 = self.backbone.layer1(x)
        x2 = self.backbone.layer2(x1)
        x3 = self.backbone.layer3(x2)  # 局部特征层
        x4 = self.backbone.layer4(x3)  # 高层特征

        # 全局特征：最终层平均池化
        global_feat = self.backbone.avgpool(x4)
        global_feat = torch.flatten(global_feat, 1)

        # 局部特征：中间层平均池化（强纹理/结构信息）
        local_feat = self.backbone.avgpool(x3)
        local_feat = torch.flatten(local_feat, 1)

        return global_feat, local_feat

# ==================== 2. 投影头（更强）====================
class ProjectionHead(nn.Module):
    def __init__(self, input_dim=2048, hidden_dim=2048, output_dim=128):
        super().__init__()
        self.mlp = nn.Sequential(
            nn.Linear(input_dim, hidden_dim),
            nn.BatchNorm1d(hidden_dim),
            nn.ReLU(inplace=True),
            nn.Linear(hidden_dim, hidden_dim),
            nn.BatchNorm1d(hidden_dim),
            nn.ReLU(inplace=True),
            nn.Linear(hidden_dim, hidden_dim),
            nn.BatchNorm1d(hidden_dim),
            nn.ReLU(inplace=True),
            nn.Linear(hidden_dim, output_dim),
            nn.BatchNorm1d(output_dim)
        )

    def forward(self, x):
        return self.mlp(x)

# ==================== 3. 全局+局部双分支 SimCLR 模型 ====================
class SimCLRModel(nn.Module):
    def __init__(self, backbone, proj_global, proj_local):
        super().__init__()
        self.encoder = backbone
        self.proj_global = proj_global
        self.proj_local = proj_local

    def forward(self, x):
        # 同时提取全局 + 局部特征
        global_feat, local_feat = self.encoder(x)

        # 投影 + 归一化
        z_global = self.proj_global(global_feat)
        z_global = nn.functional.normalize(z_global, dim=1)

        z_local = self.proj_local(local_feat)
        z_local = nn.functional.normalize(z_local, dim=1)

        return z_global, z_local, global_feat

# ==================== 4. 更强数据增强 ====================
class SimCLRTransform:
    def __init__(self, image_size):
        color_jitter = transforms.ColorJitter(0.6, 0.6, 0.6, 0.2)
        self.pil_train_transform = transforms.Compose([
            transforms.RandomResizedCrop(size=image_size, scale=(0.2, 1.0)),
            transforms.RandomHorizontalFlip(p=0.5),
            transforms.RandomApply([color_jitter], p=0.8),
            transforms.RandomGrayscale(p=0.2),
            transforms.RandomApply([transforms.GaussianBlur(3)], p=0.4),
        ])
        self.to_tensor_normalize = transforms.Compose([
            transforms.ToTensor(),
            transforms.Normalize(mean=[0.507, 0.487, 0.441],
                                std=[0.267, 0.256, 0.276]),
        ])

    def __call__(self, x):
        from torchvision.transforms.functional import to_pil_image
        if isinstance(x, torch.Tensor):
            x = to_pil_image(x)
        v1 = self.pil_train_transform(x)
        v2 = self.pil_train_transform(x)
        return self.to_tensor_normalize(v1), self.to_tensor_normalize(v2)

# 读取数据集
class CIFAR100WithClassAsCluster(torch.utils.data.Dataset):
    def __init__(self, root, train=True, transform=None, download=True):
        self.base_dataset = datasets.CIFAR100(root=root, train=train, download=download)
        self.transform = transform

    def __len__(self):
        return len(self.base_dataset)

    def __getitem__(self, idx):
        img, label = self.base_dataset[idx]
        if self.transform is not None:
            v1, v2 = self.transform(img)
        else:
            v1 = v2 = img
        return v1, v2, label, idx

# ==================== 5. 负样本损失 ====================
def hard_negative_clustered_simclr_loss(z1, z2, cluster_labels_batch, temperature=0.2, hard_neg_weight=1.6):
    device = z1.device
    batch_size = z1.shape[0]
    z_full = torch.cat((z1, z2), dim=0)
    sim_matrix_full = torch.mm(z_full, z_full.T) / temperature

    # 正样本对 mask
    pos_mask = torch.zeros((2 * batch_size, 2 * batch_size), dtype=torch.bool, device=device)
    pos_mask[torch.arange(batch_size), torch.arange(batch_size) + batch_size] = True
    pos_mask[torch.arange(batch_size) + batch_size, torch.arange(batch_size)] = True
    mask_neg = torch.ones((2 * batch_size, 2 * batch_size), dtype=torch.bool, device=device)
    mask_neg.fill_diagonal_(False)
    mask_neg[pos_mask] = False

    # 不同类别 = 合法负样本
    cl_labels_expanded = cluster_labels_batch.repeat(2)
    valid_neg_mask = (cl_labels_expanded.unsqueeze(0) != cl_labels_expanded.unsqueeze(1))
    final_neg_mask = mask_neg & valid_neg_mask

    # 硬负样本挖掘
    hard_neg_threshold = 0.45
    hard_neg_mask = (sim_matrix_full > hard_neg_threshold) & final_neg_mask

    # 加权
    weight_matrix = torch.ones_like(sim_matrix_full, device=device)
    weight_matrix[hard_neg_mask] = hard_neg_weight
    sim_matrix_weighted = sim_matrix_full * weight_matrix

    # 无效位置设为 -inf
    sim_matrix_final = sim_matrix_weighted.masked_fill(~final_neg_mask, float('-inf'))

    # 正样本相似度
    pos_sim = torch.sum(z1 * z2, dim=1) / temperature
    pos_sim_full = torch.cat([pos_sim, pos_sim], dim=0)

    # 损失计算
    exp_sim_neg = torch.exp(sim_matrix_final)
    sum_exp_neg = exp_sim_neg.sum(dim=1)
    exp_pos_sim = torch.exp(pos_sim_full)

    log_prob_pos = pos_sim_full - torch.log(exp_pos_sim + sum_exp_neg + 1e-8)
    loss = -log_prob_pos.mean()

    return loss

# ==================== 6. 全局 + 局部 联合损失 ====================
def combined_clustered_loss(z1_g, z2_g, z1_l, z2_l, cluster_labels, temp=0.22, alpha=0.55):
    loss_global = hard_negative_clustered_simclr_loss(z1_g, z2_g, cluster_labels, temp)
    loss_local  = hard_negative_clustered_simclr_loss(z1_l, z2_l, cluster_labels, temp)
    total = alpha * loss_global + (1 - alpha) * loss_local
    return total, loss_global, loss_local

def get_cosine_scheduler(optimizer, epochs, warmup_epochs=10):
    def lr_lambda(epoch):
        if epoch < warmup_epochs:
            return epoch / warmup_epochs
        return 0.5 * (1 + np.cos(np.pi * (epoch - warmup_epochs) / (epochs - warmup_epochs)))
    return optim.lr_scheduler.LambdaLR(optimizer, lr_lambda)

def plot_loss_curves(all_losses, save_path='loss_curves.png'):
    plt.figure(figsize=(10,4))
    plt.plot(all_losses)
    plt.title('Training Loss')
    plt.xlabel('Epoch')
    plt.ylabel('Loss')
    plt.grid(True)
    plt.tight_layout()
    plt.savefig(save_path, dpi=200)
    plt.close()

# ==================== 7. 更强线性评估 ====================
def evaluate_linear(model, train_loader, test_loader, num_classes=100, epochs=60, device='cuda'):
    model.eval()
    for p in model.parameters():
        p.requires_grad = False

    head = nn.Sequential(
        nn.Linear(model.encoder.feature_dim, 2048),
        nn.BatchNorm1d(2048),
        nn.ReLU(inplace=True),
        nn.Dropout(0.45),
        nn.Linear(2048, 1024),
        nn.BatchNorm1d(1024),
        nn.ReLU(inplace=True),
        nn.Dropout(0.3),
        nn.Linear(1024, num_classes)
    ).to(device)

    opt = optim.SGD(head.parameters(), lr=0.08, momentum=0.9, weight_decay=1e-4)
    scheduler = optim.lr_scheduler.CosineAnnealingLR(opt, T_max=epochs)
    criterion = nn.CrossEntropyLoss(label_smoothing=0.05)

    best = 0.0
    for ep in range(epochs):
        head.train()
        for x,y in train_loader:
            x,y = x.to(device), y.to(device)
            with torch.no_grad():
                _, _, f = model(x)
            loss = criterion(head(f), y)
            opt.zero_grad()
            loss.backward()
            opt.step()

        head.eval()
        correct, total = 0,0
        with torch.no_grad():
            for x,y in test_loader:
                x,y = x.to(device), y.to(device)
                _, _, f = model(x)
                pred = head(f).argmax(1)
                correct += (pred==y).sum().item()
                total += y.size(0)
        acc = 100.*correct/total
        if acc>best: best=acc
        print(f"Linear Epoch {ep+1:2d} | Acc={acc:.2f}% | Best={best:.2f}%")
        scheduler.step()
    return best

# ==================== 8. 主函数 ====================
def main():
    device = 'cuda' if torch.cuda.is_available() else 'cpu'
    print(f"Using device: {device}")

    batch_size = 512
    lr = 1.2e-3
    temperature = 0.22
    epochs = 100
    image_size = 32

    train_dataset = CIFAR100WithClassAsCluster(
        root='./data', train=True, transform=SimCLRTransform(image_size), download=True
    )
    train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True,
                              num_workers=2, pin_memory=True, drop_last=True)

    # 模型：双投影头
    backbone = ResNetBackbone(torchvision.models.resnet50)
    proj_global = ProjectionHead(input_dim=2048)
    proj_local  = ProjectionHead(input_dim=1024)
    model = SimCLRModel(backbone, proj_global, proj_local).to(device)

    # 优化
    optimizer = optim.AdamW(model.parameters(), lr=lr, weight_decay=1e-6)
    scheduler = get_cosine_scheduler(optimizer, epochs)
    scaler = torch.cuda.amp.GradScaler()

    # 训练
    model.train()
    all_losses = []
    for epoch in range(epochs):
        total_loss = 0.0
        pbar = tqdm(train_loader, desc=f"Epoch {epoch+1}/{epochs}")
        for v1, v2, cls_label, _ in pbar:
            v1, v2, cls_label = v1.to(device), v2.to(device), cls_label.to(device)
            optimizer.zero_grad()

            with torch.cuda.amp.autocast():
                z1_g, z1_l, _ = model(v1)
                z2_g, z2_l, _ = model(v2)
                loss, loss_g, loss_l = combined_clustered_loss(
                    z1_g, z2_g, z1_l, z2_l, cls_label, temperature
                )

            scaler.scale(loss).backward()
            scaler.step(optimizer)
            scaler.update()
            total_loss += loss.item()
            pbar.set_postfix(loss=loss.item(), g=loss_g.item(), l=loss_l.item())

        scheduler.step()
        avg_loss = total_loss / len(train_loader)
        all_losses.append(avg_loss)
        print(f"Epoch {epoch+1} Avg Loss: {avg_loss:.3f}")

    plot_loss_curves(all_losses)

    # 测试
    eval_transform = transforms.Compose([
        transforms.ToTensor(),
        transforms.Normalize([0.507,0.487,0.441],[0.267,0.256,0.276])
    ])
    lt = DataLoader(datasets.CIFAR100('./data',train=True,transform=eval_transform), batch_size=512, shuffle=True, num_workers=2)
    lv = DataLoader(datasets.CIFAR100('./data',train=False,transform=eval_transform), batch_size=512, shuffle=False, num_workers=2)

    best_acc = evaluate_linear(model, lt, lv, epochs=60, device=device)
    print(f"\nFINAL BEST ACCURACY: {best_acc:.2f}%")

if __name__ == "__main__":
    main()

Using device: cuda
Files already downloaded and verified


Epoch 1/100: 100%|██████████| 97/97 [00:42<00:00,  2.28it/s, g=7.01, l=6.75, loss=6.9] 


Epoch 1 Avg Loss: 6.888


Epoch 2/100: 100%|██████████| 97/97 [00:42<00:00,  2.26it/s, g=5.19, l=5.12, loss=5.16]


Epoch 2 Avg Loss: 5.724


...

Epoch 98 Avg Loss: 3.139


Epoch 99/100: 100%|██████████| 97/97 [00:42<00:00,  2.26it/s, g=3.1, l=3.15, loss=3.12] 


Epoch 99 Avg Loss: 3.140


Epoch 100/100: 100%|██████████| 97/97 [00:42<00:00,  2.26it/s, g=3.08, l=3.15, loss=3.11]


Epoch 100 Avg Loss: 3.140
Linear Epoch  1 | Acc=69.87% | Best=69.87%
Linear Epoch  2 | Acc=71.93% | Best=71.93%
Linear Epoch  3 | Acc=71.56% | Best=71.93%
Linear Epoch  4 | Acc=72.70% | Best=72.70%
Linear Epoch  5 | Acc=72.39% | Best=72.70%
Linear Epoch  6 | Acc=73.60% | Best=73.60%
Linear Epoch  7 | Acc=74.01% | Best=74.01%
Linear Epoch  8 | Acc=73.51% | Best=74.01%
Linear Epoch  9 | Acc=73.59% | Best=74.01%
Linear Epoch 10 | Acc=74.20% | Best=74.20%
Linear Epoch 11 | Acc=73.65% | Best=74.20%
Linear Epoch 12 | Acc=74.29% | Best=74.29%
Linear Epoch 13 | Acc=73.82% | Best=74.29%
Linear Epoch 14 | Acc=74.08% | Best=74.29%
Linear Epoch 15 | Acc=74.23% | Best=74.29%
Linear Epoch 16 | Acc=74.45% | Best=74.45%
Linear Epoch 17 | Acc=74.11% | Best=74.45%
Linear Epoch 18 | Acc=74.30% | Best=74.45%
Linear Epoch 19 | Acc=74.11% | Best=74.45%
Linear Epoch 20 | Acc=74.33% | Best=74.45%
Linear Epoch 21 | Acc=74.24% | Best=74.45%
Linear Epoch 22 | Acc=74.69% | Best=74.69%
Linear Epoch 23 | Acc=74.40%